# Batch 4 — Pre-Scan Summary




In [ ]:
import pandas as pd
from pathlib import Path
csv_path = Path(r'../data/batch4_pre_scan_summary.csv').resolve()
try:
    df = pd.read_csv(csv_path)
    df
except Exception:
    print('Failed to load', csv_path)


,filename,sex,reference_image,sample_area_cm2,bone_area_cm2,total_weight_g,soft_weight_g,lean_weight_g,fat_weight_g,fat_percent,BMC_g,BMD_mg_per_cm2
0,B4_M_0.txt,Male,B4_M_0,28.294,8.537,30.9198,30.2638,22.4686,7.7952,25.757,0.65603,76.848
1,B4_M_1.txt,Male,B4_M_1,29.147,8.066,28.8613,28.2915,21.9804,6.3111,22.308,0.56982,70.640
2,B4_M_2.txt,Male,B4_M_2,28.783,7.878,29.0044,28.4931,21.6176,6.8755,24.130,0.51129,64.897
3,B4_M_3.txt,Male,B4_M_3,30.433,8.195,31.7800,31.1786,23.2844,7.8942,25.319,0.60142,73.387
4,B4_M_4.txt,Male,B4_M_4,27.666,8.171,28.5871,27.9929,20.6969,7.2960,26.064,0.59423,72.722
5,B4_F_0.txt,Female,B4_F_0,27.806,8.235,26.7628,26.2274,16.8711,9.3563,35.674,0.53533,65.009
6,B4_F_1.txt,Female,B4_F_1,32.264,9.396,32.2373,31.5965,18.0281,13.5684,42.943,0.64086,68.209
7,B4_F_2.txt,Female,B4_F_2,24.749,7.734,21.0691,20.5939,14.7968,5.7971,28.150,0.47524,61.450
8,B4_F_3.txt,Female,B4_F_3,26.925,8.201,23.2227,22.6667,14.8896,7.7771,34.311,0.55602,67.802
9,B4_F_4.txt,Female,B4_F_4,27.246,8.737,24.5133,23.9371,15.9627,7.9744,33.314,0.57623,65.953


## Batch 4 — 1-week post-treatment summary


In [ ]:
# Load week-1 master CSV if present, otherwise scan for week-1 TXT files and build one; then display rows for this batch
from pathlib import Path
import re

DATA_CSV = Path(r'../data/week1_reports.csv').resolve()
batch_name = 'Batch 4'

def scan_and_build(downloads_root):
    week1_re = re.compile(r'(?:week[\s_-]*1|1[\s_-]*week|wk[\s_-]*1|week1|1week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week1_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

try:
    import pandas as pd
    if DATA_CSV.exists():
        master = pd.read_csv(DATA_CSV)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows = scan_and_build(downloads_root)
        master = pd.DataFrame(rows)
        try:
            master.to_csv(DATA_CSV, index=False)
        except Exception:
            pass
    if not master.empty:
        df_batch = master[master['batch'].str.lower()==batch_name.lower()]
        if not df_batch.empty:
            display(df_batch.reset_index(drop=True))
        else:
            print('No week-1 rows for', batch_name)
    else:
        print('No week-1 files found anywhere')
except Exception as e:
    print('Error building/displaying week-1 table:', e)


,batch,sex,filename,path,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 4,Female,B4_F_0.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.908,30.587,8.509,10.353,25.7996,28.8308,...,15.2672,16.3346,9.9704,11.6088,39.506,41.544,0.56193,0.88733,66.038,85.709
1,Batch 4,Female,B4_F_1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,31.119,34.171,9.189,11.059,31.1613,34.8464,...,17.9656,19.6109,12.5781,14.2881,41.181,42.149,0.61760,0.94743,67.211,85.671
2,Batch 4,Female,B4_F_2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,24.246,26.689,7.844,9.447,21.7705,24.5661,...,14.2864,15.3036,6.9663,8.4487,32.779,35.570,0.51778,0.81375,66.011,86.143
3,Batch 4,Female,B4_F_3.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,25.535,28.440,8.071,9.983,23.3732,26.6200,...,15.4278,16.8248,7.4024,8.9100,32.424,34.622,0.54302,0.88524,67.284,88.676
4,Batch 4,Female,B4_F_4.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,28.142,31.382,8.397,10.586,25.4659,28.9773,...,16.3423,17.7118,8.5632,10.3387,34.383,36.858,0.56041,0.92677,66.736,87.547
5,Batch 4,Male,B4_M_0.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,29.345,32.491,8.639,10.287,30.3955,34.0048,...,20.9336,22.6731,8.8021,10.3716,29.601,31.387,0.65981,0.96018,76.374,93.339
6,Batch 4,Male,B4_M_1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,28.023,31.782,7.727,9.787,28.8155,32.6036,...,19.8787,21.7152,8.3547,9.9700,29.592,31.466,0.58210,0.91838,75.331,93.833
7,Batch 4,Male,B4_M_2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.150,30.933,7.520,9.742,28.0261,32.3350,...,20.7295,23.1330,6.7831,8.3405,24.654,26.500,0.51352,0.86149,68.283,88.428
8,Batch 4,Male,B4_M_3.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,26.852,31.068,8.318,10.432,25.8143,29.3576,...,19.4515,21.3228,5.7661,7.1086,22.865,25.003,0.59669,0.92624,71.739,88.791
9,Batch 4,Male,B4_M_4.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,26.448,30.958,7.909,10.086,25.8286,29.9455,...,19.6689,22.1846,5.5963,6.8575,22.150,23.612,0.56347,0.90339,71.245,89.569


## Batch 4 — 2-week post-treatment summary



In [1]:
# Batch 4 — 2-week post-treatment summary
# Scan/build week-2 master CSV and display rows for this batch
from pathlib import Path
import re

DATA_CSV2 = Path(r'../data/week2_reports.csv').resolve()
batch_name = 'Batch 4'

def scan_and_build_week(downloads_root, week_num):
    week_re = re.compile(rf'(?:week[\s_-]*{week_num}|{week_num}[\s_-]*week|wk[\s_-]*{week_num}|week{week_num}|{week_num}week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

try:
    import pandas as pd
    if DATA_CSV2.exists():
        master2 = pd.read_csv(DATA_CSV2)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows2 = scan_and_build_week(downloads_root, 2)
        master2 = pd.DataFrame(rows2)
        try:
            master2.to_csv(DATA_CSV2, index=False)
        except Exception:
            pass
    if not master2.empty:
        df_batch2 = master2[master2['batch'].str.lower()==batch_name.lower()]
        if not df_batch2.empty:
            display(df_batch2.reset_index(drop=True))
        else:
            print('No week-2 rows for', batch_name)
    else:
        print('No week-2 files found anywhere')
except Exception as e:
    print('Error building/displaying week-2 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 4,Female,B4_F_0.txt,26.689,31.854,8.625,12.937,25.6237,30.9142,25.0528,...,15.7172,18.1640,9.3355,11.1129,37.264,37.958,0.57099,1.63737,66.202,126.566
1,Batch 4,Female,B4_F_1.txt,30.041,35.807,8.964,13.509,31.2565,37.3804,30.6526,...,19.0753,21.8366,11.5774,13.8543,37.770,38.817,0.60388,1.68950,67.365,125.064
2,Batch 4,Female,B4_F_2.txt,23.566,29.106,8.217,12.898,21.9399,27.2940,21.3218,...,15.0631,17.5246,6.2587,8.0480,29.354,31.471,0.61811,1.72140,75.225,133.458
3,Batch 4,Female,B4_F_3.txt,25.686,31.490,8.048,12.676,22.7915,28.3866,22.2577,...,15.6172,18.1112,6.6405,8.6381,29.835,32.293,0.53377,1.63732,66.326,129.166
4,Batch 4,Female,B4_F_4.txt,26.311,31.931,8.724,13.367,25.0039,30.7749,24.3917,...,17.0262,19.7705,7.3655,9.2969,30.197,31.984,0.61221,1.70749,70.172,127.743
5,Batch 4,Male,B4_M_0.txt,25.666,31.652,9.228,14.000,25.7549,31.9164,25.0341,...,19.6803,22.1392,5.3537,7.9317,21.386,26.377,0.72084,1.84547,78.111,131.823
6,Batch 4,Male,B4_M_1.txt,26.895,33.375,8.404,13.271,26.0216,31.8314,25.4148,...,20.9087,23.9002,4.5061,6.2550,17.730,20.743,0.60680,1.67613,72.207,126.304
7,Batch 4,Male,B4_M_2.txt,26.898,33.232,7.696,12.157,27.2206,33.5258,26.7056,...,20.6491,24.0828,6.0565,7.9134,22.679,24.732,0.51503,1.52963,66.922,125.823
8,Batch 4,Male,B4_M_4.txt,27.015,33.390,7.995,12.879,26.2959,32.4633,25.7533,...,21.0276,24.1407,4.7257,6.7169,18.350,21.767,0.54263,1.60569,67.870,124.672


## Batch 4 — 3-week post-treatment summary



In [2]:
# Batch 4 — 3-week post-treatment summary
# Scan/build week-3 master CSV and display rows for this batch (same logic as week-2)
from pathlib import Path
import re

DATA_CSV3 = Path(r'../data/week3_reports.csv').resolve()
batch_name = 'Batch 4'

try:
    import pandas as pd
    if DATA_CSV3.exists():
        master3 = pd.read_csv(DATA_CSV3)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows3 = scan_and_build_week(downloads_root, 3)
        master3 = pd.DataFrame(rows3)
        try:
            master3.to_csv(DATA_CSV3, index=False)
        except Exception:
            pass
    if not master3.empty:
        df_batch3 = master3[master3['batch'].str.lower()==batch_name.lower()]
        if not df_batch3.empty:
            display(df_batch3.reset_index(drop=True))
        else:
            print('No week-3 rows for', batch_name)
    else:
        print('No week-3 files found anywhere')
except Exception as e:
    print('Error building/displaying week-3 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 4,Female,B4_F_0.txt,26.496,30.659,8.477,11.890,25.2075,30.1213,24.6429,...,15.1493,16.8745,9.4935,11.8320,38.524,41.217,0.56460,1.41471,66.603,118.983
1,Batch 4,Female,B4_F_1.txt,31.421,36.064,8.999,12.601,30.9633,36.4341,30.3748,...,19.2253,21.1761,11.1495,13.7990,36.707,39.454,0.58847,1.45903,65.389,115.783
2,Batch 4,Female,B4_F_2.txt,24.238,28.499,8.590,12.061,22.0124,26.7612,21.3329,...,14.9766,17.0611,6.3564,8.1351,29.796,32.287,0.67943,1.56495,79.093,129.752
3,Batch 4,Female,B4_F_3.txt,25.438,29.825,8.125,11.633,22.6787,27.5633,22.1173,...,15.6981,17.6725,6.4192,8.4468,29.023,32.339,0.56147,1.44397,69.103,124.128
4,Batch 4,Female,B4_F_4.txt,27.156,31.524,8.743,12.183,24.9934,30.0888,24.3659,...,17.0017,18.9042,7.3642,9.6690,30.223,33.839,0.62751,1.51561,71.776,124.407
5,Batch 4,Male,B4_M_0.txt,26.099,30.383,8.814,12.162,26.1148,30.9153,25.4344,...,19.9778,22.3034,5.4566,7.2230,21.454,24.463,0.68046,1.38887,77.201,114.200
6,Batch 4,Male,B4_M_1.txt,27.292,32.374,8.251,11.976,27.7709,33.1014,27.1557,...,21.0892,24.0863,6.0665,7.5568,22.340,23.881,0.61521,1.45840,74.563,121.776
7,Batch 4,Male,B4_M_2.txt,28.168,32.727,8.307,11.620,28.7643,34.1996,28.2355,...,21.0193,23.4995,7.2161,9.3598,25.557,28.484,0.52881,1.34031,63.661,115.346
8,Batch 4,Male,B4_M_4.txt,27.393,32.319,7.752,11.536,26.7546,32.3820,26.2460,...,20.5249,22.6765,5.7212,8.3147,21.798,26.829,0.50854,1.39078,65.603,120.564
